# Practice: Text Data Preparation for LLMs

Large language models work better when the text you send them is clean, structured, and split into useful pieces. In this practice notebook, you will prepare small text records using the Python basics from the previous sections.

## What you will practice

- Variables, strings, lists, dictionaries, and booleans
- Indexing and slicing text
- Iterating over lists of records
- Using `if`, `continue`, `enumerate()`, `zip()`, and list comprehensions
- Building small prompt-ready text chunks with metadata

## 1. Start with raw text records

LLM data often starts as messy text from notes, tickets, chat logs, documentation, or forms. We will represent each record as a dictionary inside a list.

In [1]:
raw_documents = [
    {
        "id": "doc-001",
        "source": "course-notes",
        "text": "  Python lists keep items in order.\nUse append() to add a new item.  ",
    },
    {
        "id": "doc-002",
        "source": "support-ticket",
        "text": "Email support@example.com if the notebook kernel fails.",
    },
    {
        "id": "doc-003",
        "source": "empty-note",
        "text": "     ",
    },
    {
        "id": "doc-004",
        "source": "lesson-summary",
        "text": "Loops help prepare text. For each document, clean whitespace, check length, and split it into chunks.",
    },
]

## 2. Inspect the data types and a small preview

Before cleaning data, inspect its structure. This uses `type()`, indexing, dictionary keys, and string slicing.

In [2]:
first_document = raw_documents[0]

print(type(raw_documents))
print(type(first_document))
print(first_document["id"])
print(first_document["text"][:40])  # Preview the first 40 characters.

<class 'list'>
<class 'dict'>
doc-001
  Python lists keep items in order.
Use 


## 3. Clean one text string first

A useful pattern is to test your cleaning steps on one example before applying them to every record.

In [3]:
sample_text = raw_documents[0]["text"]

clean_text = sample_text.strip()
clean_text = " ".join(clean_text.split())  # Collapse repeated spaces and newlines.

print(clean_text)

Python lists keep items in order. Use append() to add a new item.


## 4. Clean and filter all records

Now apply the same idea to every document. We will skip empty records, normalize whitespace, and remove a sample email address before the text is used with an LLM.

In [4]:
cleaned_documents = []
minimum_length = 20

for document in raw_documents:
    text = document["text"].strip()
    text = " ".join(text.split())
    text = text.replace("support@example.com", "[EMAIL]")  # Simple redaction.

    if len(text) < minimum_length:
        continue  # Skip records that are too short to be useful.

    cleaned_document = {
        "id": document["id"],
        "source": document["source"],
        "text": text,
        "length": len(text),
    }
    cleaned_documents.append(cleaned_document)

print(cleaned_documents)

[{'id': 'doc-001', 'source': 'course-notes', 'text': 'Python lists keep items in order. Use append() to add a new item.', 'length': 65}, {'id': 'doc-002', 'source': 'support-ticket', 'text': 'Email [EMAIL] if the notebook kernel fails.', 'length': 43}, {'id': 'doc-004', 'source': 'lesson-summary', 'text': 'Loops help prepare text. For each document, clean whitespace, check length, and split it into chunks.', 'length': 101}]


## 5. Review cleaned records with `enumerate()` and slicing

When you have many records, print short previews instead of full text. This is where indexing and slicing become practical.

In [5]:
for position, document in enumerate(cleaned_documents, start=1):
    preview = document["text"][:70]
    print(f"{position}. {document['id']} ({document['source']}): {preview}...")

1. doc-001 (course-notes): Python lists keep items in order. Use append() to add a new item....
2. doc-002 (support-ticket): Email [EMAIL] if the notebook kernel fails....
3. doc-004 (lesson-summary): Loops help prepare text. For each document, clean whitespace, check le...


## 6. Split text into small chunks

LLMs have context limits, so long text is usually split into chunks. This example uses words as a simple approximation. Real applications often use a tokenizer, but the loop logic is the same idea.

In [6]:
chunks = []
chunk_size = 8

for document in cleaned_documents:
    words = document["text"].split()

    for start in range(0, len(words), chunk_size):
        word_chunk = words[start:start + chunk_size]
        chunk_text = " ".join(word_chunk)

        chunks.append({
            "document_id": document["id"],
            "chunk_id": len(chunks) + 1,
            "text": chunk_text,
            "word_count": len(word_chunk),
        })

print(chunks)

[{'document_id': 'doc-001', 'chunk_id': 1, 'text': 'Python lists keep items in order. Use append()', 'word_count': 8}, {'document_id': 'doc-001', 'chunk_id': 2, 'text': 'to add a new item.', 'word_count': 5}, {'document_id': 'doc-002', 'chunk_id': 3, 'text': 'Email [EMAIL] if the notebook kernel fails.', 'word_count': 7}, {'document_id': 'doc-004', 'chunk_id': 4, 'text': 'Loops help prepare text. For each document, clean', 'word_count': 8}, {'document_id': 'doc-004', 'chunk_id': 5, 'text': 'whitespace, check length, and split it into chunks.', 'word_count': 8}]


## 7. Pair labels with chunks using `zip()`

`zip()` is useful when two sequences should be read together. Here we pair display labels with the first few chunks.

In [7]:
selected_chunks = chunks[:3]
labels = ["Context A", "Context B", "Context C"]

for label, chunk in zip(labels, selected_chunks):
    print(f"{label}: {chunk['text']}")

Context A: Python lists keep items in order. Use append()
Context B: to add a new item.
Context C: Email [EMAIL] if the notebook kernel fails.


## 8. Build prompt-ready context

A common LLM workflow is to attach a small amount of cleaned context to a question. Keeping source metadata makes the result easier to trace.

In [8]:
context_lines = []

for chunk in selected_chunks:
    source_label = f"[{chunk['document_id']}:{chunk['chunk_id']}]"
    context_lines.append(f"{source_label} {chunk['text']}")

prompt_context = "\n".join(context_lines)
question = "How should I prepare text before sending it to an LLM?"

print("Context:")
print(prompt_context)
print("\nQuestion:")
print(question)

Context:
[doc-001:1] Python lists keep items in order. Use append()
[doc-001:2] to add a new item.
[doc-002:3] Email [EMAIL] if the notebook kernel fails.

Question:
How should I prepare text before sending it to an LLM?


## 9. Run lightweight quality checks

Before using prepared text, check basic assumptions: no empty chunks, expected metadata is present, and useful text remains after cleaning.

In [9]:
chunk_texts = [chunk["text"] for chunk in chunks]
short_chunks = [chunk for chunk in chunks if chunk["word_count"] < chunk_size]

has_empty_text = any(text == "" for text in chunk_texts)
all_have_document_id = all("document_id" in chunk for chunk in chunks)

print(f"Any empty chunks? {has_empty_text}")
print(f"Every chunk has a document id? {all_have_document_id}")
print(f"Short final chunks: {len(short_chunks)}")

Any empty chunks? False
Every chunk has a document id? True
Short final chunks: 2


# Practice exercises

Use the same data-preparation ideas below. The starter cells are intentionally incomplete but runnable, so you can fill them in step by step.

## Exercise 1: Clean a list of text snippets

Clean each string by stripping whitespace, collapsing repeated spaces, replacing the sample email with `[EMAIL]`, and skipping empty results.

In [10]:
exercise_texts = [
    "  Prompt engineering needs clean inputs.  ",
    "",
    "LLM data should not include private emails like teacher@example.com.",
]

cleaned_exercise_texts = []

for text in exercise_texts:
    # TODO: clean the text and append only valid results.
    pass

print(cleaned_exercise_texts)

[]


## Exercise 2: Chunk a document

Split the document into chunks of six words. Store each chunk as a string in `exercise_chunks`.

In [11]:
exercise_document = "LLMs work better when the input text is clean organized and connected to useful metadata."
exercise_chunk_size = 6
exercise_chunks = []

# TODO: split exercise_document into words and create chunks of exercise_chunk_size words.

print(exercise_chunks)

[]


## Exercise 3: Combine text with metadata using `zip()`

Use `zip()` to combine each topic with its matching text. Store dictionaries with the keys `topic` and `text`.

In [12]:
topics = ["cleaning", "filtering", "chunking"]
texts = [
    "Remove unnecessary whitespace.",
    "Skip records that are empty or too short.",
    "Split long text into smaller pieces.",
]

training_records = []

# TODO: use zip(topics, texts) to build dictionaries inside training_records.

print(training_records)

[]


## Exercise 4: Mini project

Prepare the `raw_notes` list for an LLM. Clean the text, skip weak records, split useful records into chunks, and keep the original note id as metadata.

In [13]:
raw_notes = [
    {"id": "note-1", "text": "  Lists and dictionaries are useful for storing structured records.  "},
    {"id": "note-2", "text": "short"},
    {"id": "note-3", "text": "Use loops and conditions to prepare text before sending it to a language model."},
]

prepared_records = []

# TODO: combine cleaning, filtering, chunking, and metadata into prepared_records.

print(prepared_records)

[]


## Challenge exercise: Build a retrieval-ready mini dataset

Create a small pipeline that prepares messy notes for an LLM retrieval workflow. This exercise combines variables, strings, lists, dictionaries, slicing, loops, conditions, `continue`, `enumerate()`, `zip()`, and list comprehensions.

Your tasks:

1. Clean each note by stripping whitespace and collapsing repeated spaces.
2. Redact email addresses from the text.
3. Skip notes that are empty, too short, or marked as low priority.
4. Split valid notes into chunks of at most `challenge_chunk_size` words.
5. Add metadata to every chunk: note id, topic, priority, chunk number, and word count.
6. Build a quality report with the number of skipped notes, created chunks, and topics included.
7. Print a prompt-ready context using the first three chunks.

In [14]:
challenge_notes = [
    {
        "id": "kb-001",
        "topic": "loops",
        "priority": "high",
        "text": "  Use for loops when you already have a collection. Use while loops when you need to repeat until a condition changes.  ",
    },
    {
        "id": "kb-002",
        "topic": "privacy",
        "priority": "high",
        "text": "Contact admin@example.com before sharing notebooks that contain student data.",
    },
    {
        "id": "kb-003",
        "topic": "draft",
        "priority": "low",
        "text": "temporary note",
    },
    {
        "id": "kb-004",
        "topic": "chunking",
        "priority": "medium",
        "text": "Chunking keeps long context manageable. Each chunk should keep enough words to be useful and enough metadata to be traceable.",
    },
    {
        "id": "kb-005",
        "topic": "empty",
        "priority": "high",
        "text": "     ",
    },
]

challenge_chunk_size = 7
minimum_words = 5
challenge_chunks = []
skipped_notes = []

for note in challenge_notes:
    # TODO 1: clean whitespace in note["text"].
    clean_note_text = note["text"]

    # TODO 2: redact admin@example.com with [EMAIL].

    # TODO 3: split the cleaned text into words.
    words = []

    # TODO 4: skip low-priority notes, empty notes, and notes with fewer than minimum_words words.


    # TODO 5: create chunks of challenge_chunk_size words and append dictionaries to challenge_chunks.


topics_included = []  # TODO 6: build this from challenge_chunks.
quality_report = {
    "skipped_notes": len(skipped_notes),
    "created_chunks": len(challenge_chunks),
    "topics_included": topics_included,
}

print("Quality report:")
print(quality_report)

print("\nPrompt-ready context:")
for number, chunk in enumerate(challenge_chunks[:3], start=1):
    # TODO 7: print each chunk with its note id, topic, and text.
    pass

Quality report:
{'skipped_notes': 0, 'created_chunks': 0, 'topics_included': []}

Prompt-ready context:


## Reflection questions

- Which records should be removed before using the data with an LLM?
- What metadata would help you trace an answer back to the source?
- When would a list comprehension make the code clearer, and when would a normal loop be easier to read?
- What could go wrong if you chunk text without keeping the document id?